In [ ]:
# 1/4 - clone CUDA backend branch (HEAD de1941b: 3 kernels .tkv)
import subprocess, os
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
print('cu exists', os.path.exists(d+'/Release/cuda_backend/examples/fused_ops.cuda.cu'))

In [ ]:
# 2/4 - build .so from compiler-generated .cu
import subprocess, shutil, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
cu=ex+'/fused_ops.cuda.cu'; so=ex+'/fused_ops.so'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
cmd=(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -std=c++20 -Xcompiler -fPIC '+cu+' -o '+so+' -lcudart')
p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
print('build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:3000])
print('so exists',os.path.exists(so))

In [ ]:
# 3/4 - test 3 kernels vs PyTorch
import ctypes, torch, numpy as np
so='/content/TokenVectorCuda/Release/cuda_backend/examples/fused_ops.so'
lib=ctypes.CDLL(so); I32=ctypes.c_int; I64=ctypes.c_int64; F32=ctypes.c_float
lib.tkv_launch_embed_add.argtypes=[I64,I64,I64,I64,I32,I32,I32,I32]; lib.tkv_launch_embed_add.restype=I32
lib.tkv_launch_gelu_erf.argtypes=[I64,I64,I32,I32,I32]; lib.tkv_launch_gelu_erf.restype=I32
lib.tkv_launch_bias_add.argtypes=[I64,I64,I64,I32,I32,I32,I32]; lib.tkv_launch_bias_add.restype=I32
T=128; H=64; n=T*H
wte=torch.randn(50257,H,device='cuda',dtype=torch.float32)
wpe=torch.randn(1024,H,device='cuda',dtype=torch.float32)
ids=torch.randint(0,50257,(T,),device='cuda',dtype=torch.int32)
out=torch.zeros(T,H,device='cuda',dtype=torch.float32)
rc=lib.tkv_launch_embed_add(wte.data_ptr(),wpe.data_ptr(),ids.data_ptr(),out.data_ptr(),n,H,(n+255)//256,256)
ref=wte[ids.long()]+wpe[torch.arange(T,device='cuda')]
print('embed_add rc=',rc,' maxerr=',(out-ref).abs().max().item())
x=torch.randn(4096,device='cuda',dtype=torch.float32); g=torch.zeros_like(x)
rc=lib.tkv_launch_gelu_erf(x.data_ptr(),g.data_ptr(),x.numel(),(x.numel()+255)//256,256)
refg=0.5*x*(1.0+torch.erf(x*0.70710678))  # erf gelu, direct math
print('gelu_erf rc=',rc,' maxerr=',(g-refg).abs().max().item())
a=torch.randn(T,H,device='cuda',dtype=torch.float32); b=torch.randn(H,device='cuda',dtype=torch.float32); o=torch.zeros_like(a)
rc=lib.tkv_launch_bias_add(a.data_ptr(),b.data_ptr(),o.data_ptr(),n,H,(n+255)//256,256)
print('bias_add rc=',rc,' maxerr=',(o-(a+b)).abs().max().item())

In [ ]:
# 4/4 - summary: if maxerr small, kernels correct (but fp16 mismatch noted)
print('DONE. Note: kernels accept fp32 pointers; forward uses fp16 -> need conversion or f16 DSL support.')